# MMM Framework — Quickstart

Fit your first Bayesian Marketing Mix Model in a few lines, using a **bundled example dataset** (no data wrangling required), then **grade the estimate against a sealed answer key**.

Runs top-to-bottom in a few minutes on a free Colab CPU. See the full [Getting Started guide](https://redam94.github.io/mmm-framework/getting-started.html).

In [ ]:
# Install the framework. Until the next PyPI release ships load_example,
# install from source; afterwards `pip install mmm-framework` is enough.
!pip install -q "mmm-framework @ git+https://github.com/redam94/mmm-framework.git"

## Fit a model and read off the ROI

In [ ]:
from mmm_framework import (
    load_example,
    load_example_answer_key,
    BayesianMMM,
    ModelConfigBuilder,
    TrendConfig,
    TrendType,
)

# 1. Load a bundled example — 104 weeks of national weekly data, ready to model.
panel = load_example("national")
print(panel.summary())

# 2. Configure inference (fast JAX/NumPyro sampler) and a linear trend.
model_config = (
    ModelConfigBuilder()
    .bayesian_numpyro()      # ~3x faster than PyMC at equal draws
    .with_chains(4)
    .with_draws(500)         # small + fast for a first run
    .with_tune(500)
    .build()
)
trend_config = TrendConfig(type=TrendType.LINEAR)

# 3. Fit. On a laptop this national model takes roughly 15-25 seconds.
mmm = BayesianMMM(panel, model_config, trend_config)
results = mmm.fit(random_seed=42)
print("max R-hat:", round(results.diagnostics["rhat_max"], 3))   # ~1.0 = converged

# 4. The headline: each channel's return on ad spend (contribution / spend).
decomp = mmm.compute_component_decomposition()
roi = (decomp.media_by_channel.sum() / panel.X_media.sum()).sort_values(ascending=False)
print("\nEstimated ROI by channel:")
print(roi.round(2))

# 5. This example ships a SEALED answer key — grade the estimate against truth.
truth = load_example_answer_key("national")["true_roas"]
print("\nchannel   estimated   true")
for ch in roi.index:
    print(f"  {ch:<8} {roi[ch]:>8.2f}   {truth[ch]:>5.2f}")

The model recovers the causal **ranking** (brand channels TV / Social / Video out-earn performance channels Search / Display), even though a single observational fit attenuates the magnitudes. Tightening those magnitudes is what the framework's [experiment-calibration loop](https://redam94.github.io/mmm-framework/measurement-calibration.html) is for.

**Next:** swap in your own data ([MFF format](https://redam94.github.io/mmm-framework/getting-started.html#mff-format)), or explore the [learning series](https://redam94.github.io/mmm-framework/demos.html).